In [1]:
import pandas as pd

In [2]:
from sklearn import set_config
set_config(display="text")

In [3]:
data = pd.read_csv("../data/online_shoppers.csv")

In [4]:
data.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [5]:
df = data.drop_duplicates()
df.shape

(12205, 18)

In [6]:
y = df['Revenue']
X= df.drop(columns = 'Revenue')


In [7]:
from sklearn.model_selection import train_test_split

In [8]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify= y)

In [9]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9764, 17), (2441, 17), (9764,), (2441,))

In [10]:
y_train.mean()

np.float64(0.15628840639082342)

In [11]:
y_test.mean()

np.float64(0.15649324047521507)

## Split
total size after duplicated drop = (12205,18)
test rate = 0.2
Purchase rate: train 15.6%, test 15.6% (stratified)

In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

In [13]:
dummy = DummyClassifier(strategy = 'most_frequent')
dummy.fit(X_train,y_train)
predictions = dummy.predict(X_train)

print(f" Accuracy : {accuracy_score(y_train, predictions):.3f} \n Recall : {recall_score(y_train,predictions)}")


 Accuracy : 0.844 
 Recall : 0.0


## Dummy baseline
- The dummy (most_frequent) always predicts majority class for every session.
- Accuracy = 84.4% because 84.4% of sessions are non-buyers, so "always no" is right that often.
- Recall = 0.0% because it never predicts minority class, so it catches none of the real buyers.
- Conclusion: accuracy is misleading on imbalanced data; we use PR-AUC instead.

## Logistic regression pipeline

In [14]:
X_train.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000
mean,2.342687,82.219938,0.510344,34.950623,32.101803,1207.264687,0.020296,0.041469,5.945606,0.062351,2.130582,2.371876,3.134986,4.059402
std,3.349268,179.502363,1.282728,143.828323,44.817312,1926.389159,0.045234,0.046145,18.860076,0.199959,0.909649,1.734820,2.389418,4.026181
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,8.000000,193.466667,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,9.000000,0.000000,0.000000,18.000000,617.000000,0.002857,0.025000,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,94.725000,0.000000,0.000000,39.000000,1483.650000,0.016667,0.049242,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


In [15]:
numeric_cols = ['Administrative', 'Administrative_Duration','Informational','Informational_Duration','ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues','SpecialDay']
categorical_cols = ['OperatingSystems','Browser','Region','TrafficType','Weekend','Month','VisitorType']
len(numeric_cols + categorical_cols)

17

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [17]:
preprocessor = ColumnTransformer([("num",StandardScaler(),numeric_cols),("category",OneHotEncoder(handle_unknown= 'ignore', sparse_output=False),categorical_cols)])


In [18]:
logreg_pipe = Pipeline([("prep", preprocessor), ("lr_model", LogisticRegression())])

In [19]:
logreg_pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'BounceRates', 'ExitRates',
                                                   'PageValues',
                                                   'SpecialDay']),
                                                 ('category',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
             

In [20]:
logreg_pipe.predict_proba(X_train.head())

array([[0.17766248, 0.82233752],
       [0.42140132, 0.57859868],
       [0.94732716, 0.05267284],
       [0.92838859, 0.07161141],
       [0.95462574, 0.04537426]])

In [21]:
y_train.head()


7556      True
11924     True
3306     False
207      False
4460     False
Name: Revenue, dtype: bool

In [22]:
logreg_pipe.classes_


array([False,  True])

## Logistic regression summary
- Structure: 10 numeric columns are scaled with mean and std, 7 categorical columns are one-hot encoded with 64 features. In total 74 features going into model.
- Unknown categories are ignored (`handle_unknown='ignore'`). Example: TrafficType 12 (1 session) is absent from train.
- Fitted on the training set only.
- `predict_proba` returns 2 columns in the order of `classes_` = [False, True]: column 0 = No purchase, column 1 = Purchase. Each row sums to 1.
- We use column 1 for PR-AUC, because PR-AUC measures how well the model ranks the positive class at the top.
- Sanity check on 5 training rows.

In [23]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 42)

In [24]:
lr_cv = cross_validate(logreg_pipe, X_train, y_train, cv = cv, scoring= "average_precision")
lr_cv

{'fit_time': array([0.03319073, 0.02380681, 0.03198719, 0.030931  , 0.02209711]),
 'score_time': array([0.00417995, 0.00361013, 0.00375199, 0.00358915, 0.00355482]),
 'test_score': array([0.650092  , 0.59438053, 0.63246486, 0.64764865, 0.65318198])}

In [25]:
print(f" logistic regression + cross validation: \n mean: {lr_cv["test_score"].mean():.3f} \n standard deviation : {lr_cv['test_score'].std():.3f}")

 logistic regression + cross validation: 
 mean: 0.636 
 standard deviation : 0.022


In [26]:
dummy_cv = cross_validate(dummy, X_train, y_train, cv = cv, scoring= "average_precision")
print(dummy_cv['test_score'].mean())
print(dummy_cv['test_score'].std())


0.1562884024577573
0.00019923684608644864


In [27]:
rows = [{'model' : 'Dummy (most frequent)', 'pr_auc_mean' : dummy_cv['test_score'].mean().round(3), 'pr_auc_std':dummy_cv['test_score'].std().round(3)},
        {'model' : 'Logistic Regression', 'pr_auc_mean' : lr_cv["test_score"].mean().round(3), 'pr_auc_std':lr_cv['test_score'].std().round(3)}]

In [28]:
comparison_table = pd.DataFrame(rows)

In [29]:
comparison_table

,model,pr_auc_mean,pr_auc_std
0,Dummy (most frequent),0.156,0.000
1,Logistic Regression,0.636,0.022


## Cross-validation :
- 5-fold stratified CV on the training set, metric = PR-AUC (average precision)
- Logistic regression 0.636 ± 0.022, x4 better than dummy (0.156)

-----------------------------------------------------------------------------------

## Gradient Boosting

In [30]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [31]:
hgb_pipe = Pipeline([("prep", preprocessor), ("hgb_model", HistGradientBoostingClassifier(random_state= 42))])

In [32]:
hgb_cv = cross_validate(hgb_pipe, X_train, y_train, cv = cv, scoring= "average_precision")

In [33]:
print(hgb_cv['test_score'].mean())
print(hgb_cv['test_score'].std())

0.7423994655620276
0.022444269839407883


In [34]:
rows.append({'model' : 'Gradient boosting', 'pr_auc_mean' : hgb_cv['test_score'].mean().round(3), 'pr_auc_std':hgb_cv['test_score'].std().round(3)})

In [35]:
comparison_table = pd.DataFrame(rows)
comparison_table

,model,pr_auc_mean,pr_auc_std
0,Dummy (most frequent),0.156,0.000
1,Logistic Regression,0.636,0.022
2,Gradient boosting,0.742,0.022


## Gradient boosting results:
- Same preprocessing and CV folds; only the model changed (HistGradientBoostingClassifier, defaults).
- PR-AUC 0.742 ± 0.022

## Tuning (gradient boosting)

In [36]:
from sklearn.model_selection import GridSearchCV

In [37]:
hgb_pipe['hgb_model']

HistGradientBoostingClassifier(random_state=42)

In [38]:
grid_params = {
    "hgb_model__learning_rate" : [0.05, 0.1, 0.2],
    "hgb_model__max_leaf_nodes" : [21, 31, 41],
    "hgb_model__min_samples_leaf" : [10, 20, 30]
}

In [39]:
grid_search = GridSearchCV(hgb_pipe, grid_params, cv = cv, scoring= 'average_precision')

In [40]:
grid_search.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['Administrative',
                                                                          'Administrative_Duration',
                                                                          'Informational',
                                                                          'Informational_Duration',
                                                                          'ProductRelated',
                                                                          'ProductRelated_Duration',
                                                                          'BounceRates',
                                       

In [41]:
grid_search.best_params_


{'hgb_model__learning_rate': 0.05,
 'hgb_model__max_leaf_nodes': 21,
 'hgb_model__min_samples_leaf': 20}

In [42]:
grid_search.best_score_

np.float64(0.7568666958555393)

## results:
it increased the score from 0.742 to 0.757 with params of (learning rate = 0.05,
max_leaf_nodes = 21, min_samples_leaf = 20), which is smaller than CV spread 
(±0.022), it's small improvement and could partly be noise.

## Without PageValues

In [43]:
numeric_cols_no_pv = [col for col in numeric_cols if col != "PageValues"]
len(numeric_cols_no_pv)

9

In [44]:
preprocessor_no_pv = ColumnTransformer([("num",StandardScaler(),numeric_cols_no_pv),("category",OneHotEncoder(handle_unknown= 'ignore', sparse_output=False),categorical_cols)])

In [45]:
logreg_pipe_no_pv = Pipeline([("prep", preprocessor_no_pv), ("lr_model", LogisticRegression())])

In [46]:
hgb_pipe_no_pv = Pipeline([("prep", preprocessor_no_pv), ("hgb_model", HistGradientBoostingClassifier(random_state= 42))])

In [47]:
lr_cv_no_pv = cross_validate(logreg_pipe_no_pv, X_train, y_train, cv = cv, scoring= "average_precision")
lr_cv_no_pv

{'fit_time': array([0.04256701, 0.02780294, 0.03284097, 0.03091669, 0.0294261 ]),
 'score_time': array([0.00436711, 0.00507879, 0.00401211, 0.00378919, 0.00382185]),
 'test_score': array([0.3368909 , 0.27910792, 0.36051542, 0.32692316, 0.33555949])}

In [48]:
hgb_cv_no_pv = cross_validate(hgb_pipe_no_pv, X_train, y_train, cv = cv, scoring= "average_precision")
hgb_cv_no_pv

{'fit_time': array([0.50869465, 0.53243089, 0.52377009, 0.49681091, 0.48341489]),
 'score_time': array([0.01048017, 0.01086998, 0.00938606, 0.00886989, 0.0079062 ]),
 'test_score': array([0.3592909 , 0.3473847 , 0.39507653, 0.36924749, 0.3553648 ])}

In [49]:
rows.extend([
    {'model': 'Logistic regression (no PageValues)', 'pr_auc_mean': lr_cv_no_pv['test_score'].mean().round(3), 'pr_auc_std': lr_cv_no_pv['test_score'].std().round(3)},
    {'model': 'Gradient boosting (no PageValues)',   'pr_auc_mean': hgb_cv_no_pv['test_score'].mean().round(3), 'pr_auc_std': hgb_cv_no_pv['test_score'].std().round(3)}
])

In [50]:
rows

[{'model': 'Dummy (most frequent)',
  'pr_auc_mean': np.float64(0.156),
  'pr_auc_std': np.float64(0.0)},
 {'model': 'Logistic Regression',
  'pr_auc_mean': np.float64(0.636),
  'pr_auc_std': np.float64(0.022)},
 {'model': 'Gradient boosting',
  'pr_auc_mean': np.float64(0.742),
  'pr_auc_std': np.float64(0.022)},
 {'model': 'Logistic regression (no PageValues)',
  'pr_auc_mean': np.float64(0.328),
  'pr_auc_std': np.float64(0.027)},
 {'model': 'Gradient boosting (no PageValues)',
  'pr_auc_mean': np.float64(0.365),
  'pr_auc_std': np.float64(0.016)}]

In [51]:
comparison_table = pd.DataFrame(rows)
comparison_table

,model,pr_auc_mean,pr_auc_std
0,Dummy (most frequent),0.156,0.000
1,Logistic Regression,0.636,0.022
2,Gradient boosting,0.742,0.022
3,Logistic regression (no PageValues),0.328,0.027
4,Gradient boosting (no PageValues),0.365,0.016


## Without PageValues results:
- without PageValues scores decrease significantly, LR 0.636 -> 0.328, HGB 0.742 -> 0.365.
- this shows dependence, which is consistent with the leakage hypothesis but not proof of it, without PageValues the realistic performance is about 0.36

-----------------------------------------------------------------------------------

## Threshold and final test evaluation

Use case: the shop shows an offer (e.g. a discount voucher or free shipping) to hesitant visitors during the session. Visitors the model scores as likely buyers get no discount, because they would buy anyway.

Decided before opening the test set:
- Models evaluated: HGB tuned, with PageValues; HGB default, without PageValues.
- Business rule: score >= threshold means likely buyer, no discount. Score below the threshold means show the discount.
- Threshold rule: for each model, the lowest threshold with precision >= 0.8 (business assumption), chosen on train CV predictions.
- Why precision: a non-buyer wrongly called "buyer" gets no discount and may leave (lost sale); a missed buyer only gets an unneeded discount (some margin lost).
- Limits: the score is not a calibrated probability; the discount effect would need an A/B test.

In [52]:
from sklearn.model_selection import cross_val_predict

In [53]:
grid_search.best_estimator_

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'BounceRates', 'ExitRates',
                                                   'PageValues',
                                                   'SpecialDay']),
                                                 ('category',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
             

In [54]:
scores_pv = cross_val_predict(hgb_pipe, X_train, y_train, cv=cv, method="predict_proba")[:,1]
scores_pv.shape

(9764,)

In [55]:
scores_no_pv = cross_val_predict(hgb_pipe_no_pv, X_train, y_train, cv=cv, method="predict_proba")[:,1]
scores_no_pv.shape

(9764,)

In [56]:
scores_pv[1:10].round(3), y_train.iloc[1:10]

(array([0.988, 0.001, 0.001, 0.   , 0.64 , 0.457, 0.043, 0.06 , 0.001]),
 11924     True
 3306     False
 207      False
 4460     False
 6786     False
 1277     False
 5921     False
 6562     False
 962      False
 Name: Revenue, dtype: bool)

In [57]:
from sklearn.metrics import precision_recall_curve

In [58]:
precision, recall, threshold = precision_recall_curve(y_train, scores_pv)


In [59]:
precision, recall, threshold

(array([0.15628841, 0.15630441, 0.15632043, ..., 1.        , 1.        ,
        1.        ], shape=(9707,)),
 array([1.00000000e+00, 1.00000000e+00, 1.00000000e+00, ...,
        1.31061599e-03, 6.55307995e-04, 0.00000000e+00], shape=(9707,)),
 array([8.53521803e-05, 9.02500570e-05, 9.19319166e-05, ...,
        9.95259546e-01, 9.96681976e-01, 9.96763569e-01], shape=(9706,)))

In [60]:
mask = precision[:-1]>=0.8


In [61]:
import numpy as np

idx = np.argmax(mask)

In [62]:
print(threshold[idx].round(3), precision[idx].round(3),recall[idx].round(3))

0.684 0.8 0.443


In [63]:
precision_no_pv, recall_no_pv, thresholds_no_pv = precision_recall_curve(y_train, scores_no_pv)

In [64]:
mask_no_pv = precision_no_pv[:-1] >= 0.8

In [65]:
mask_no_pv.sum()

np.int64(3)

In [66]:
idx_no_pv = np.argmax(mask_no_pv)

In [67]:
recall_no_pv[idx_no_pv].round(3)

np.float64(0.002)

## Threshold results (train CV predictions)
- With PageValues: threshold 0.684, precision 0.8, recall 0.443. When we require 80% of flagged visitors to be real buyers, the score threshold is 0.684, and we catch only 44.3% of all buyers.
- Without PageValues: precision >= 0.8 only at recall 0.002, so the model cannot pick out buyers with 80% precision; in practice it would show the offer to almost everyone.
- Decision for the test: PR-AUC for both models; precision, recall and confusion matrix at 0.684 for the PageValues model only (possible leakage, see above). No threshold for the no-PageValues model.

In [68]:
hgb_model = hgb_pipe.fit(X_train,y_train)
hgb_no_pv_model = hgb_pipe_no_pv.fit(X_train,y_train)

In [69]:
test_scores_pv = hgb_model.predict_proba(X_test)[:,1]
test_scores_no_pv = hgb_no_pv_model.predict_proba(X_test)[:,1]

In [70]:
print(test_scores_pv.shape, test_scores_no_pv.shape)

(2441,) (2441,)


In [71]:
test_scores_pv[:5].round(3)

array([0.023, 0.016, 0.046, 0.069, 0.   ])

In [72]:
from sklearn.metrics import average_precision_score

In [73]:
average_precision_score(y_test, test_scores_pv)

0.7409947670539658

In [74]:
average_precision_score(y_test, test_scores_no_pv)

0.3698509789395192

In [75]:
t_pv = threshold[idx]
t_pv

np.float64(0.683680711207736)

In [76]:
test_pred_pv = test_scores_pv >= t_pv
test_pred_pv.sum()

np.int64(224)

In [77]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

In [78]:
precision_score(y_test, test_pred_pv)

0.7901785714285714

In [79]:
recall_score(y_test, test_pred_pv)

0.46335078534031415

In [80]:
confusion_matrix(y_test,test_pred_pv)

array([[2012,   47],
       [ 205,  177]])

## Test results (test set used once)
- PR-AUC: with PageValues 0.741 (CV 0.742), without 0.370 (CV 0.365). Test matches CV, so the CV estimates were reliable on unseen data from the same dataset. This does not rule out PageValues leakage, which would affect train and test equally.
- With PageValues at threshold 0.684: precision 0.790, recall 0.463.
- Confusion matrix : 2012 non-buyers correctly get an offer, 177 buyers correctly get no offer; 47 non-buyers wrongly get no offer; 205 buyers still get an unneeded offer.
- Without PageValues the model is not usable for this rule (see threshold results).
- Limits: possible PageValues leakage; score not calibrated; offer effect not measured (needs A/B test).